# Phase 7 — Prior Case Retrieval — Model 6 — LoRA

Run this notebook from top to bottom in a fresh kernel with the matching foundation checkpoint and tokenizer. The experiment is already selected in the configuration cell.

**Source:** `phase7/capstone-p7-all-models-all-methods.ipynb` at commit `7e7c8a9f1605c5f9836a3cb2cb7c28004343f5f3`. Model/training/evaluation/submission implementations are copied unchanged; only configuration selections and output directories differ. Saved outputs and execution counts were cleared during organization.

**Outputs:** use a separate Kaggle session or working directory for each run. The original submission filenames are preserved, so some ZIP files at the working-directory root can be overwritten by another run.

## Original implementation notes

# Phase 7 - Prior Case Retrieval (PCR) with your own pre-trained models (Model 4 / 6 / 7)
## Bi-encoder with four adaptation methods: **full**, **lora**, **lorma**, **knowledge_distillation**

Same retrieval skeleton as the earlier Phase-7 notebook (long case -> up to `CHUNKS` windows -> encoder -> window pooling -> L2-normalised case vector; InfoNCE with in-batch + TF-IDF hard negatives; hybrid dense + TF-IDF scoring; exactly 7 candidates per test query). The encoder is **one of your own pre-trained models**, picked with `MODEL_OPTION` in Section 2 (same options and architectures as the Phase-6 notebook):

| `MODEL_OPTION` | Architecture |
|---|---|
| `4` | Model 4 - `EfficientLegalLM`, 16 layers, 768d (run bidirectionally, as in Phase 3/4) |
| `6` | Model 6 - `EfficientLegalLM`, 20 layers, 1,024d (run bidirectionally) |
| `7` | Model 7 - 12-layer BERT-style stack, 768d, uncased WordPiece (`BertModel`) |

Only the selected model is loaded. No external pre-trained model is used anywhere.

| `RUN_METHODS` entry | Meaning |
|---|---|
| `'full'` | all weights of the selected model are fine-tuned |
| `'lora'` | additive low-rank adapters on every linear layer, base frozen |
| `'lorma'` | multiplicative output-space adapters (Phase 3/4/5 definition), base frozen |
| `'knowledge_distillation'` (`'kd'` alias) | a fully fine-tuned **teacher** (reused from `'full'` when it was run, otherwise trained automatically) provides target embeddings / score distributions; a **6-layer student** initialised from evenly spaced pre-trained layers is trained on InfoNCE + distillation. Teacher and student never share the GPU (teacher embeddings are cached on CPU first) |

Model selection uses **dev only**; test labels are never touched. `prediction.json` / `submission.zip` are rewritten every time a candidate (single model or ensemble) beats the best dev Micro-F1@7 so far, so a session that dies half-way still leaves a valid submission.

**Attach as Kaggle inputs:** the checkpoint dataset (`best.pt`) of the selected model and its tokenizer dataset. This notebook has only been syntax-checked, not executed.

## 1. Install and imports

In [ ]:
%pip install -q "transformers>=4.44,<5" "huggingface_hub>=0.24,<2" "scikit-learn>=1.3" "safetensors>=0.4"

In [ ]:
import os, json, math, random, re, gc, time, zipfile, glob
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint
from tqdm.auto import tqdm
from transformers import (AutoTokenizer, PreTrainedTokenizerFast, BertTokenizerFast, BertConfig, BertModel,
                          get_linear_schedule_with_warmup)
from huggingface_hub import hf_hub_download
from sklearn.feature_extraction.text import TfidfVectorizer
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## 2. Configuration

In [ ]:
# ======================= what to run =======================
MODEL_OPTION = 6                 # <<< CHOOSE which of your pre-trained models: 4 | 6 | 7
RUN_METHODS = ['lora']           # any subset of: 'full', 'lora', 'lorma', 'knowledge_distillation'  ('kd' is accepted as an alias)
USE_HYBRID = True                # add TF-IDF cosine to the dense score (weight tuned on dev)
RETRAIN = True                   # False -> reuse best_<name>_pcr.pt from OUTPUT_DIR (or CKPT_OVERRIDE) when it exists
CKPT_OVERRIDE = {}               # e.g. {'m4_lorma': '/kaggle/input/x/best_m4_lorma_pcr.pt', 'm4_kd': '...', 'm4_kd_teacher': '...'}

REPRESENTATION = 'head_tail'     # 'uniform' | 'head_tail'  (which windows of a long case are kept)
POOLING = 'attention'            # window aggregation: 'attention' | 'mean'
WINDOW_POOL = 'mean'             # inside a window: 'mean' (masked token mean) | 'cls' (first token)

# ======================= your pre-trained model (checkpoint + tokenizer) =======================
CKPT_PATH = None                 # set to force a specific best.pt (otherwise the default below / auto-found under /kaggle/input)
TOK_DIR = None                   # set to force a tokenizer folder (otherwise the default below / auto-found under /kaggle/input)
DEFAULT_PATHS = {                # optional known Kaggle paths per model: (best.pt, tokenizer folder); used only if they exist
    4: ('/kaggle/input/datasets/paveetsaluja/capstone-m4-ckpts/best.pt',
        '/kaggle/input/datasets/paveetsaluja/m4-dataset-500m/legal_tokenizer_efficient'),
}

# ===================== architecture of the selected model (same as the Phase-6 notebook) =====================
assert MODEL_OPTION in (4, 6, 7), 'MODEL_OPTION must be 4, 6 or 7'
if MODEL_OPTION in (4, 6):
    ARCH = 'lm'                 # EfficientLegalLM (RMSNorm, RoPE, GQA, SwiGLU, alternating window/full layers)
    if MODEL_OPTION == 4:
        D_MODEL, N_LAYERS, N_HEADS, N_KV_HEADS, D_FF, WINDOW_SIZE = 768, 16, 12, 2, 2048, 256
    else:
        D_MODEL, N_LAYERS, N_HEADS, N_KV_HEADS, D_FF, WINDOW_SIZE = 1024, 20, 16, 4, 2432, 512
    HEAD_DIM, MAX_SEQ_LEN, ROPE_BASE = D_MODEL // N_HEADS, 1024, 10000.0
else:
    ARCH = 'bert'               # Model 7: BERT-base-style stack, prefix-LM pre-trained
    M7_CFG = dict(vocab_size=30522, hidden_size=768, num_hidden_layers=12, num_attention_heads=12,
                  intermediate_size=3072, hidden_act='gelu', max_position_embeddings=512, type_vocab_size=2,
                  layer_norm_eps=1e-12, hidden_dropout_prob=0.1, attention_probs_dropout_prob=0.1,
                  position_embedding_type='absolute')
    D_MODEL, MAX_SEQ_LEN = M7_CFG['hidden_size'], M7_CFG['max_position_embeddings']
    CKPT_KEY_PREFIX = ''        # optional extra prefix to strip from checkpoint keys (after '_orig_mod.' / 'module.' / 'bert.')
MODEL_NAME = f'Model {MODEL_OPTION}'
PFX = f'm{MODEL_OPTION}'         # prefix of every checkpoint / score file written by this run

BIDIRECTIONAL = True             # Models 4/6: bidirectional attention at fine-tune time (as in Phase 3)
BIDIR_WINDOW  = 256              # Models 4/6: width of the sliding-window layers when run bidirectionally
RANK, ALPHA   = 8, 16.0          # LoRA / LoRMA
USE_GC        = True             # gradient checkpointing (needed for 'full' on a T4; turn off for lora/lorma if memory allows)
STUDENT_LAYERS = 6               # knowledge_distillation: student keeps this many evenly spaced pre-trained layers
KD_HARD_WEIGHT = 0.5             # knowledge_distillation: loss = w * InfoNCE + (1-w) * (KL on score distributions + cosine to teacher embeddings)

# ======================= data =======================
HF_DATASET_REPO = 'Exploration-Lab/CS-781-Capstone'
P7 = 'Phase_7_PCR'
HF_TOKEN = os.environ.get('HF_TOKEN')
OUTPUT_DIR = Path('/kaggle/working/pcr_p7_model6_lora' if Path('/kaggle/working').exists() else '/mnt/data/pcr_p7_model6_lora')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ======================= text / windows =======================
SEED = 42; SMOKE = False
TOPK = 7
MAX_LEN = 384; CHUNKS = 6; STRIDE = 320
MAX_CHARS = 150000               # very long cases keep head+tail characters only
LEX_CHARS = 60000                # chars per doc used for TF-IDF

# ======================= training =======================
BATCH_Q = 6                      # queries per step -> docs per step = BATCH_Q*(1+HN_PER_Q); lower this on OOM
HN_PER_Q = 1
HN_POOL_K = 50
GRAD_ACCUM = 1
EPOCHS = 20; PATIENCE = 4
TEMP = 0.05
HEAD_LR = 1e-3                   # attention-pooling head
LR_BACKBONE = {'full': 5e-5, 'lora': 3e-4, 'lorma': 3e-4}   # backbone LR per method (Phase 3/4 presets); the KD student is trained like 'full'
WEIGHT_DECAY = 0.01; WARMUP_FRAC = 0.06
TIME_BUDGET_MIN = None           # e.g. 180 caps the training minutes of every model; each model also stops by EPOCHS/PATIENCE
INFER_DOCS = 16

RUN_METHODS = ['knowledge_distillation' if m == 'kd' else m for m in RUN_METHODS]
METHOD_ORDER = ['full', 'lora', 'lorma', 'knowledge_distillation']   # run order ('full' first so its checkpoint can serve as the KD teacher)
assert RUN_METHODS and len(set(RUN_METHODS)) == len(RUN_METHODS) and set(RUN_METHODS) <= set(METHOD_ORDER)
assert REPRESENTATION in ('uniform', 'head_tail') and POOLING in ('attention', 'mean') and WINDOW_POOL in ('mean', 'cls')
assert CHUNKS >= 3 and STRIDE <= MAX_LEN - 2 and MAX_LEN <= MAX_SEQ_LEN
assert 0 <= KD_HARD_WEIGHT <= 1 and 1 <= STUDENT_LAYERS < (N_LAYERS if ARCH == 'lm' else M7_CFG['num_hidden_layers'])
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = DEVICE.type == 'cuda'
print(f'{MODEL_NAME} ({ARCH}) | methods {RUN_METHODS} | {DEVICE} | {OUTPUT_DIR}')

## 3. Download Phase 7 data, validate schema, build per-split query/pool structures
Each split's queries are only ever ranked against **that split's own** candidate pool.

In [ ]:
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF")
    except Exception:
        pass

def load_json(rel):
    p = hf_hub_download(repo_id=HF_DATASET_REPO, repo_type="dataset", filename=rel, token=HF_TOKEN)
    with open(p, encoding="utf-8") as f:
        return json.load(f)

def to_text(t):
    if isinstance(t, list):
        return " ".join(" ".join(map(str, s)) if isinstance(s, list) else str(s) for s in t)
    return str(t)

REL_KEYS = ("relevant_candidates", "relevant", "labels", "label")

def to_records(raw, labeled, name):
    if isinstance(raw, dict):
        for k in ("data", "examples", "queries", "candidates"):
            if isinstance(raw.get(k), list):
                raw = raw[k]; break
        else:
            raw = [dict(v, id=k) if isinstance(v, dict) else {"id": k, "text": v} for k, v in raw.items()]
    assert isinstance(raw, list), "Expected list of records or id-indexed dict"
    out = []; seen = set()
    for row in raw:
        assert isinstance(row, dict) and "id" in row and "text" in row, f"Bad row in {name}: {str(row)[:200]}"
        rid = str(row["id"])
        if rid in seen:
            continue
        seen.add(rid)
        r = {"id": rid, "text": to_text(row["text"])}
        if labeled:
            key = next((k for k in REL_KEYS if k in row), None)
            assert key, f"No relevance key in {name}:{rid}; keys={list(row)}"
            r["relevant"] = [str(x) for x in (row[key] or [])]
        out.append(r)
    print(f"{name}: {len(out):,} records (raw {len(raw):,})")
    return out

def make_split(queries, pool):
    cids = [r["id"] for r in pool]; cidx = {c: i for i, c in enumerate(cids)}
    gold = [set(r.get("relevant", [])) for r in queries]
    rows, cols = [], []
    for i, r in enumerate(queries):
        j = cidx.get(r["id"])
        if j is not None:
            rows.append(i); cols.append(j)
    return dict(queries=queries, pool=pool, cids=cids, cidx=cidx, gold=gold,
                self=(np.array(rows, dtype=np.int64), np.array(cols, dtype=np.int64)))

def read_split(folder, name, labeled):
    q = to_records(load_json(f"{P7}/{folder}/{name}.json"), labeled, f"{name} queries")
    c = to_records(load_json(f"{P7}/{folder}/{name}_candidates.json"), False, f"{name} candidates")
    return q, c

tr_q, tr_c = read_split("Train", "train", True)
dv_q, dv_c = read_split("Dev", "dev", True)
te_q, te_c = read_split("Test", "test", False)

if SMOKE:
    rs = np.random.RandomState(0)
    def shrink(q, c, nq, npool):
        q = q[:nq]; need = {x for r in q for x in r.get("relevant", [])}
        keep = [r for r in c if r["id"] in need]
        rest = [r for r in c if r["id"] not in need]
        keep += [rest[i] for i in rs.permutation(len(rest))[:npool]]
        return q, keep
    tr_q, tr_c = shrink(tr_q, tr_c, 24, 300); dv_q, dv_c = shrink(dv_q, dv_c, 16, 300)
    te_q, te_c = te_q[:8], te_c[:300]; EPOCHS = 1; print("SMOKE MODE ENABLED")

SP = {"train": make_split(tr_q, tr_c), "dev": make_split(dv_q, dv_c), "test": make_split(te_q, te_c)}

for name, sp in SP.items():
    n_pool = len(sp["pool"]); lens = [len(r["text"]) for r in sp["pool"]]
    msg = f"{name.upper():5s} queries {len(sp['queries']):,} | pool {n_pool:,} | pool median chars {int(np.median(lens)):,} | queries also in pool: {len(sp['self'][0])}"
    if name != "test":
        ng = [len(g) for g in sp["gold"]]
        inpool = sum(len(g & set(sp["cids"])) for g in sp["gold"]); tot = sum(ng)
        msg += f" | gold/query mean {np.mean(ng):.2f} median {np.median(ng):.0f} max {max(ng)} | gold found in pool {inpool}/{tot}"
        assert tot > 0
    print(msg)
    assert n_pool > TOPK + 1, "pool too small for top-7 retrieval"

# training queries: gold restricted to the train pool
GOLD_IN_POOL = {}
for r, g in zip(SP["train"]["queries"], SP["train"]["gold"]):
    gp = sorted(g & set(SP["train"]["cids"]) - {r["id"]})
    if gp:
        GOLD_IN_POOL[r["id"]] = gp
TR_Q = [r for r in SP["train"]["queries"] if r["id"] in GOLD_IN_POOL]
GOLD_SET = {r["id"]: set(GOLD_IN_POOL[r["id"]]) for r in TR_Q}
print(f"Training queries with >=1 relevant candidate in pool: {len(TR_Q):,} / {len(SP['train']['queries']):,}")
assert len(TR_Q) >= 2

## 4. Locate checkpoint and tokenizer, tokenizer setup, window selection
The tokenizer and checkpoint of the selected model are located and loaded exactly as in the Phase-6 notebook. Each case is tokenised once and at most `CHUNKS` windows are kept (`uniform`: evenly spaced; `head_tail`: first/last windows contiguous, the rest spread over the middle). Windows are wrapped with the selected model's own special tokens (`[CLS] ... [SEP]` for Model 7, `... <eos>` for Models 4/6).

In [ ]:
def _find(patterns, prefer=()):
    hits = sorted({h for p in patterns for h in glob.glob(p, recursive=True)})
    for key in prefer:
        pref = [h for h in hits if key in h.lower()]
        if pref:
            return pref[0], hits
    return (hits[0] if hits else None), hits

_dflt = DEFAULT_PATHS.get(MODEL_OPTION)
if CKPT_PATH is None and _dflt and os.path.isfile(_dflt[0]):
    CKPT_PATH = _dflt[0]
if TOK_DIR is None and _dflt and os.path.isdir(_dflt[1]) and (os.path.isfile(os.path.join(_dflt[1], 'tokenizer.json')) or os.path.isfile(os.path.join(_dflt[1], 'vocab.txt'))):
    TOK_DIR = _dflt[1]
if CKPT_PATH is None:
    prefer = {4: ('m4',), 6: ('m6',), 7: ('m7', 'model7', 'model_7')}[MODEL_OPTION] + ('capstone',)
    CKPT_PATH, hits = _find(['/kaggle/input/**/best.pt'], prefer=prefer)
    print('best.pt candidates:', hits)
if TOK_DIR is None:
    if ARCH == 'lm':
        tj, hits = _find(['/kaggle/input/**/tokenizer.json'], prefer=('legal_tokenizer', 'tokenizer'))
    else:
        tj, hits = _find(['/kaggle/input/**/vocab.txt', '/kaggle/input/**/tokenizer.json'], prefer=('m7', 'model7', 'legal', 'tokenizer'))
    TOK_DIR = os.path.dirname(tj) if tj else None
    print('tokenizer candidates:', hits)
assert CKPT_PATH and os.path.isfile(CKPT_PATH), f'best.pt for {MODEL_NAME} not found - attach its checkpoint dataset or set CKPT_PATH'
assert TOK_DIR and os.path.isdir(TOK_DIR), f'tokenizer for {MODEL_NAME} not found - attach its tokenizer dataset or set TOK_DIR'
print(f'{MODEL_NAME}: make sure these belong to {MODEL_NAME}')
print('using checkpoint:', CKPT_PATH); print('using tokenizer :', TOK_DIR)

if ARCH == 'lm':
    tokenizer = PreTrainedTokenizerFast.from_pretrained(TOK_DIR)
else:
    try:                                   # uncased WordPiece, 30,522 entries
        tokenizer = BertTokenizerFast.from_pretrained(TOK_DIR, do_lower_case=True)
    except Exception as e:
        print('BertTokenizerFast failed (', e, ') - falling back to AutoTokenizer')
        tokenizer = AutoTokenizer.from_pretrained(TOK_DIR)
tokenizer.model_max_length = 10 ** 9      # we window ourselves; silence the length warning
VOCAB_SIZE = len(tokenizer)
print(f'VOCAB_SIZE={VOCAB_SIZE} fast={tokenizer.is_fast}')

if ARCH == 'lm':
    PAD_L = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    SEP_L = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else PAD_L
    CLS_L, N_SPECIAL = None, 1             # window = tokens + <eos>
    def wrap(chunk): return list(chunk) + [SEP_L]
else:
    assert VOCAB_SIZE == M7_CFG['vocab_size'], f'tokenizer has {VOCAB_SIZE} entries but Model 7 was built for {M7_CFG["vocab_size"]}'
    assert None not in (tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id)
    PAD_L, SEP_L, CLS_L, N_SPECIAL = tokenizer.pad_token_id, tokenizer.sep_token_id, tokenizer.cls_token_id, 2
    def wrap(chunk): return [CLS_L] + list(chunk) + [SEP_L]     # window = [CLS] tokens [SEP]
    print('tokenizer lowercases:', tokenizer.tokenize('Section 302 IPC, The Supreme Court'))
print('PAD', PAD_L, 'SEP/EOS', SEP_L, 'CLS', CLS_L, '| special tokens per window:', N_SPECIAL)

def clip(text, n=MAX_CHARS):
    return text if len(text) <= n else text[: n // 2] + " " + text[-(n // 2):]

def window_starts(n, W):
    if n <= W:
        return [0]
    max_start = n - W
    allst = list(range(0, max_start + 1, STRIDE))
    if allst[-1] != max_start:
        allst.append(max_start)
    if len(allst) <= CHUNKS:
        return allst
    if REPRESENTATION == 'uniform':
        idx = np.unique(np.linspace(0, len(allst) - 1, CHUNKS).round().astype(int))
        return [allst[i] for i in idx]
    h = max(1, CHUNKS // 3); t = max(1, CHUNKS // 3); mid = CHUNKS - h - t
    head = allst[:h]; tail = allst[-t:]; mid_c = allst[h:len(allst) - t]
    chosen = list(head)
    if mid > 0 and mid_c:
        idx = np.unique(np.linspace(0, len(mid_c) - 1, mid).round().astype(int))
        chosen += [mid_c[i] for i in idx]
    chosen += tail
    return sorted(set(chosen))

class Windowizer:
    # windows are wrapped with the selected model's own special tokens (see wrap above)
    def __init__(self, tok):
        self.tok, self.cache = tok, {}
        self.W = MAX_LEN - N_SPECIAL
        self.unk = tok.unk_token_id if tok.unk_token_id is not None else PAD_L
    def make(self, ids):
        if not ids:
            ids = [self.unk]
        return [np.array(wrap(ids[s:s + self.W]), dtype=np.int32) for s in window_starts(len(ids), self.W)]
    def prefetch(self, records, kind, bs=64):
        todo = [r for r in records if (kind, r["id"]) not in self.cache]
        for i in tqdm(range(0, len(todo), bs), desc=f"tokenise {kind} ({len(todo):,} docs)", leave=False):
            batch = todo[i:i + bs]
            enc = self.tok([clip(r["text"]) for r in batch], add_special_tokens=False, truncation=False)["input_ids"]
            for r, ids in zip(batch, enc):
                self.cache[(kind, r["id"])] = self.make(ids)
    def get(self, kind, rid):
        return self.cache[(kind, rid)]

WZ = Windowizer(tokenizer)
# dev/test: everything is needed. train: queries now, candidates after hard-negative mining (Section 5).
WZ.prefetch(SP["dev"]["queries"], "dq"); WZ.prefetch(SP["dev"]["pool"], "dc")
WZ.prefetch(SP["test"]["queries"], "tq"); WZ.prefetch(SP["test"]["pool"], "tc")
WZ.prefetch(TR_Q, "q")
ex = WZ.get("dq", SP["dev"]["queries"][0]["id"])
print("example dev query windows:", len(ex), "| first window tokens:", len(ex[0]))

## 5. Metrics, lexical (TF-IDF) scores, hard-negative mining
`Micro-F1@7` is computed exactly as the task defines it: top-7 of every query vs the *full* gold set. A query never retrieves itself if it appears in its own pool.

In [ ]:
def zrow(S):
    S = np.asarray(S, dtype=np.float64)
    return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-6)

def topk_ids(S, sp, k=TOPK):
    S = np.asarray(S).astype(np.float32, copy=True)
    r, c = sp["self"]
    if len(r):
        S[r, c] = -1e9
    idx = np.argpartition(-S, k, axis=1)[:, :k]
    order = np.argsort(-np.take_along_axis(S, idx, 1), axis=1)
    return np.take_along_axis(idx, order, 1)

def score_f1(S, sp):
    top = topk_ids(S, sp); tp = 0; ng = 0
    for i, row in enumerate(top):
        g = sp["gold"][i]; tp += sum(sp["cids"][j] in g for j in row); ng += len(g)
    P = tp / (TOPK * len(top)); R = tp / max(ng, 1)
    return (2 * P * R / (P + R) if P + R > 0 else 0.0), P, R

def lexical(queries, pool, topk=None, exclude_self=True):
    vec = TfidfVectorizer(sublinear_tf=True, min_df=2, max_df=0.5, max_features=300000, stop_words="english", dtype=np.float32)
    X = vec.fit_transform([clip(r["text"], LEX_CHARS) for r in pool] + [clip(r["text"], LEX_CHARS) for r in queries])
    XP, XQ = X[:len(pool)], X[len(pool):]
    cidx = {r["id"]: i for i, r in enumerate(pool)}
    outs = []
    for i in range(0, len(queries), 256):
        blk = (XQ[i:i + 256] @ XP.T).toarray().astype(np.float32)
        if exclude_self:
            for b, r in enumerate(queries[i:i + 256]):
                j = cidx.get(r["id"])
                if j is not None:
                    blk[b, j] = -1.0
        if topk:
            idx = np.argpartition(-blk, topk, axis=1)[:, :topk]
            order = np.argsort(-np.take_along_axis(blk, idx, 1), axis=1)
            outs.append(np.take_along_axis(idx, order, 1))
        else:
            outs.append(blk)
    return np.concatenate(outs, 0)

t0 = time.time()
LEX = {}
for name in ("dev", "test"):
    LEX[name] = lexical(SP[name]["queries"], SP[name]["pool"])
f1, P, R = score_f1(LEX["dev"], SP["dev"])
print(f"TF-IDF baseline on dev: F1@7 {f1:.4f} | P@7 {P:.4f} | R@7 {R:.4f}  ({time.time()-t0:.0f}s)")
LEXZ = {k: zrow(v) for k, v in LEX.items()}

# hard negatives for training queries: top lexical candidates that are NOT relevant
top_idx = lexical(SP["train"]["queries"], SP["train"]["pool"], topk=HN_POOL_K * 2)
qpos = {r["id"]: i for i, r in enumerate(SP["train"]["queries"])}
HN = {}
for r in TR_Q:
    cand = [SP["train"]["cids"][j] for j in top_idx[qpos[r["id"]]]]
    HN[r["id"]] = [c for c in cand if c not in GOLD_SET[r["id"]] and c != r["id"]][:HN_POOL_K]
del top_idx
NEED_C = {c for r in TR_Q for c in GOLD_IN_POOL[r["id"]]} | {c for v in HN.values() for c in v}
WZ.prefetch([r for r in SP["train"]["pool"] if r["id"] in NEED_C], "c")
print(f"train candidates tokenised: {len(NEED_C):,} | avg hard-neg list {np.mean([len(v) for v in HN.values()]):.1f}")

## 6. Your pre-trained model, adapters, document encoder, model factory
Definitions of `EfficientLegalLM` (Models 4/6) and the checkpoint helpers for Model 7 are copied from your Phase 3 notebooks / the Phase-6 notebook. `make_model(method, student)` builds a fresh copy of the selected model with the requested adaptation; `student=True` keeps only `STUDENT_LAYERS` evenly spaced pre-trained layers (used by knowledge distillation).

In [ ]:
# ---------------- Models 4/6: EfficientLegalLM (identical to the Phase-1 / Phase-3 notebooks) ----------------
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * rms).type_as(x) * self.weight

def precompute_rope_freqs(dim, max_seq_len, base=10000.0):
    freqs = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    return torch.cos(freqs), torch.sin(freqs)

def apply_rope(x, freqs_cos, freqs_sin):
    x_r = x.reshape(*x.shape[:-1], -1, 2)
    x0, x1 = x_r[..., 0], x_r[..., 1]
    seq_len = x.shape[2]
    cos = freqs_cos[:seq_len].unsqueeze(0).unsqueeze(0).to(x.dtype)
    sin = freqs_sin[:seq_len].unsqueeze(0).unsqueeze(0).to(x.dtype)
    out0 = x0 * cos - x1 * sin
    out1 = x0 * sin + x1 * cos
    return torch.stack([out0, out1], dim=-1).reshape_as(x)

class SwiGLU_FFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.0):
        super().__init__()
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.w_up = nn.Linear(d_model, d_ff, bias=False)
        self.w_down = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout(self.w_down(F.silu(self.w_gate(x)) * self.w_up(x)))

def repeat_kv(x, n_rep):
    if n_rep == 1:
        return x
    B, H, T, D = x.shape
    return x[:, :, None, :, :].expand(B, H, n_rep, T, D).reshape(B, H * n_rep, T, D)

class GQAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, head_dim, dropout=0.0, use_window=False, window_size=256):
        super().__init__()
        self.n_heads = n_heads; self.n_kv_heads = n_kv_heads
        self.head_dim = head_dim; self.n_rep = n_heads // n_kv_heads
        self.use_window = use_window; self.window_size = window_size
        self.attn_p = dropout
        self.q_proj = nn.Linear(d_model, n_heads * head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * head_dim, bias=False)
        self.out_proj = nn.Linear(n_heads * head_dim, d_model, bias=False)
        self.resid_dropout = nn.Dropout(dropout)
    def forward(self, x, freqs_cos, freqs_sin, use_cache=False, past_kv=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q = apply_rope(q, freqs_cos, freqs_sin)
        k = apply_rope(k, freqs_cos, freqs_sin)
        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        new_kv = (k, v) if use_cache else None
        dp = self.attn_p if self.training else 0.0
        if past_kv is not None:
            if self.use_window:
                k = k[:, :, -self.window_size:]; v = v[:, :, -self.window_size:]
            k = repeat_kv(k, self.n_rep); v = repeat_kv(v, self.n_rep)
            out = F.scaled_dot_product_attention(q, k, v, dropout_p=dp)
        else:
            k = repeat_kv(k, self.n_rep); v = repeat_kv(v, self.n_rep)
            if self.use_window and T > 1:
                row = torch.arange(T, device=x.device).unsqueeze(1)
                col = torch.arange(T, device=x.device).unsqueeze(0)
                mask = (col <= row) & (col > row - self.window_size)
                out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=dp)
            else:
                out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=dp)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.out_proj(out)), new_kv

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, head_dim, d_ff, dropout=0.0, use_window=False, window_size=256):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = GQAttention(d_model, n_heads, n_kv_heads, head_dim, dropout, use_window, window_size)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU_FFN(d_model, d_ff, dropout)
    def forward(self, x, freqs_cos, freqs_sin, use_cache=False, past_kv=None):
        attn_out, new_kv = self.attn(self.attn_norm(x), freqs_cos, freqs_sin, use_cache, past_kv)
        x = x + attn_out
        x = x + self.ffn(self.ffn_norm(x))
        return x, new_kv

class EfficientLegalLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, n_kv_heads, head_dim, d_ff,
                 max_seq_len, rope_base=10000.0, dropout=0.0, window_size=256, use_gc=True):
        super().__init__()
        self.d_model = d_model; self.n_layers = n_layers; self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len; self.use_gc = use_gc
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.emb_dropout = nn.Dropout(dropout)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_heads, n_kv_heads, head_dim, d_ff, dropout,
                             use_window=(i % 2 == 0), window_size=window_size)
            for i in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        fc, fs = precompute_rope_freqs(head_dim, max_seq_len, rope_base)
        self.register_buffer("freqs_cos", fc, persistent=False)
        self.register_buffer("freqs_sin", fs, persistent=False)
    def forward(self, input_ids, targets=None, use_cache=False, past_kvs=None):
        B, T = input_ids.shape
        x = self.emb_dropout(self.tok_emb(input_ids))
        past_len = past_kvs[0][0].shape[2] if past_kvs is not None else 0
        fc = self.freqs_cos[past_len:past_len + T]
        fs = self.freqs_sin[past_len:past_len + T]
        presents = [] if use_cache else None
        for i, layer in enumerate(self.layers):
            pkv = past_kvs[i] if past_kvs is not None else None
            x, kv = layer(x, fc, fs, use_cache, pkv)
            if use_cache:
                presents.append(kv)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits[:, :-1, :].contiguous().view(-1, self.vocab_size),
                                   targets[:, 1:].contiguous().view(-1), ignore_index=-100)
        return logits, loss, presents

# ---------------- bidirectional blocks + adapters (identical to the Phase-3 notebook) ----------------
def build_masks(T, pad_mask, bidirectional, window):
    # boolean attention masks (True = may attend), shape (B,1,T,T): one for window layers, one for full layers
    r = torch.arange(T, device=pad_mask.device).unsqueeze(1)
    c = torch.arange(T, device=pad_mask.device).unsqueeze(0)
    if bidirectional:
        full = torch.ones(T, T, dtype=torch.bool, device=pad_mask.device)
        win = (c - r).abs() < window
    else:
        full = c <= r
        win = (c <= r) & (c > r - window)
    key = pad_mask[:, None, None, :]
    return full[None, None] & key, win[None, None] & key

def block_forward(blk, x, cos, sin, mask):
    a = blk.attn
    h = blk.attn_norm(x)
    B, T, _ = h.shape
    q = a.q_proj(h).view(B, T, a.n_heads, a.head_dim).transpose(1, 2)
    k = a.k_proj(h).view(B, T, a.n_kv_heads, a.head_dim).transpose(1, 2)
    v = a.v_proj(h).view(B, T, a.n_kv_heads, a.head_dim).transpose(1, 2)
    q = apply_rope(q, cos, sin)
    k = apply_rope(k, cos, sin)
    k = repeat_kv(k, a.n_rep)
    v = repeat_kv(v, a.n_rep)
    out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
    out = out.transpose(1, 2).contiguous().view(B, T, -1)
    x = x + a.out_proj(out)
    return x + blk.ffn(blk.ffn_norm(x))

class LoRALinear(nn.Module):
    # y = W x + (alpha/r) B A x   (base W frozen, B starts at zero)
    def __init__(self, base_linear, rank, alpha):
        super().__init__()
        self.base = base_linear
        out_f, in_f = base_linear.weight.shape
        self.A = nn.Parameter(torch.randn(rank, in_f) * 0.02)
        self.B = nn.Parameter(torch.zeros(out_f, rank))
        self.scale = alpha / rank
    def forward(self, x):
        return self.base(x) + self.scale * ((x @ self.A.t()) @ self.B.t())

class LoRMALinear(nn.Module):
    # LoRMA (multiplicative): y = (I + (alpha/r) B A) W x   (base W frozen, B starts at zero -> identical to Phase 1 at step 0)
    def __init__(self, base_linear, rank, alpha):
        super().__init__()
        self.base = base_linear
        out_f = base_linear.weight.shape[0]
        self.A = nn.Parameter(torch.randn(rank, out_f) * 0.02)
        self.B = nn.Parameter(torch.zeros(out_f, rank))
        self.scale = alpha / rank
    def forward(self, x):
        h = self.base(x)
        return h + self.scale * ((h @ self.A.t()) @ self.B.t())

def adapt_linears(backbone, kind, rank, alpha):
    wrapper = LoRMALinear if kind == "lorma" else LoRALinear
    n = 0
    for blk in backbone.layers:
        for parent in (blk.attn, blk.ffn):
            for name, child in list(parent.named_children()):
                if isinstance(child, nn.Linear):
                    setattr(parent, name, wrapper(child, rank, alpha))
                    n += 1
    return n


def adapt_bert(encoder, method, rank, alpha):
    # same placement as the Phase-6 notebook: q/k/v, attention output, intermediate and output dense of every BERT layer
    wrapper = LoRMALinear if method == "lorma" else LoRALinear
    n = 0
    for layer in encoder.encoder.layer:
        for parent, names in [(layer.attention.self, ['query', 'key', 'value']), (layer.attention.output, ['dense']),
                              (layer.intermediate, ['dense']), (layer.output, ['dense'])]:
            for name in names:
                lin = getattr(parent, name)
                assert isinstance(lin, nn.Linear), (name, type(lin))
                setattr(parent, name, wrapper(lin, rank, alpha)); n += 1
    return n

# ---------------- Model 7: checkpoint helpers (as in the Phase-6 notebook / capstone-p3-model7) ----------------
def build_model7():
    # embeddings (token + learned position + segment, LayerNorm) -> 12 post-LN layers; pooler is not needed for retrieval
    return BertModel(BertConfig(**M7_CFG), add_pooling_layer=False)

def _norm_key(k):
    for pre in ("_orig_mod.", "module."):
        while k.startswith(pre):
            k = k[len(pre):]
    if CKPT_KEY_PREFIX and k.startswith(CKPT_KEY_PREFIX):
        k = k[len(CKPT_KEY_PREFIX):]
    return k.replace(".LayerNorm.gamma", ".LayerNorm.weight").replace(".LayerNorm.beta", ".LayerNorm.bias")

def split_checkpoint(raw):
    # return (encoder_state_dict, prediction_head_state_dict) from a Model 7 checkpoint of any common wrapping
    sd = raw
    if isinstance(raw, dict):
        for k in ("model_state_dict", "model", "state_dict", "global_state_dict", "theta_g"):
            if k in raw and isinstance(raw[k], dict):
                sd = raw[k]; break
    enc, head = {}, {}
    for k, v in sd.items():
        if not torch.is_tensor(v):
            continue
        k = _norm_key(k)
        if k.startswith("bert."):
            enc[k[len("bert."):]] = v
        elif k.startswith("cls."):
            head[k] = v
        else:
            enc[k] = v
    return enc, head

class PredHead:
    # BERT prediction (LM) head used only for the load sanity check: dense -> GELU -> LayerNorm -> tied decoder (+ bias)
    def __init__(self, head_sd, emb_w):
        g = lambda *names: next((head_sd[n] for n in names if n in head_sd), None)
        self.dw = g("cls.predictions.transform.dense.weight"); self.db = g("cls.predictions.transform.dense.bias")
        self.lw = g("cls.predictions.transform.LayerNorm.weight"); self.lb = g("cls.predictions.transform.LayerNorm.bias")
        self.bias = g("cls.predictions.bias", "cls.predictions.decoder.bias")
        self.emb_w = emb_w
        self.ok = None not in (self.dw, self.db, self.lw, self.lb)
    def __call__(self, h):
        d = F.gelu(F.linear(h, self.dw.to(h.device, h.dtype), self.db.to(h.device, h.dtype)))
        d = F.layer_norm(d, (D_MODEL,), self.lw.to(h.device, h.dtype), self.lb.to(h.device, h.dtype), M7_CFG["layer_norm_eps"])
        b = self.bias.to(h.device, h.dtype) if self.bias is not None else None
        return F.linear(d, self.emb_w.to(h.dtype), b)

# ---------------- load the pre-trained weights once (CPU) ----------------
_ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
if ARCH == 'lm':
    ENC_SD = {(k[len("_orig_mod."):] if k.startswith("_orig_mod.") else k): v for k, v in _ck["model_state_dict"].items()}   # saved from a torch.compile()d model
    HEAD_SD = None
    print("checkpoint meta:", {k: _ck.get(k) for k in ("step", "loss")})
else:
    ENC_SD, HEAD_SD = split_checkpoint(_ck)
    print("checkpoint meta:", {k: _ck[k] for k in ("step", "round", "loss", "val_loss") if isinstance(_ck, dict) and k in _ck})
del _ck

def build_backbone(keep_head=False):
    # a fresh copy of the pre-trained model, weights loaded from ENC_SD (checked every time)
    if ARCH == 'lm':
        b = EfficientLegalLM(VOCAB_SIZE, D_MODEL, N_LAYERS, N_HEADS, N_KV_HEADS, HEAD_DIM, D_FF,
                             MAX_SEQ_LEN, ROPE_BASE, 0.0, WINDOW_SIZE, use_gc=False)
        missing, unexpected = b.load_state_dict(ENC_SD, strict=False)
        missing = [k for k in missing if k != "lm_head.weight"]       # tied to tok_emb.weight
        if not keep_head:
            del b.lm_head                                             # unused for retrieval (tied to tok_emb)
    else:
        b = build_model7()
        missing, unexpected = b.load_state_dict(ENC_SD, strict=False)
        unexpected = [k for k in unexpected if k != "embeddings.position_ids" and not k.startswith("pooler.")]
    assert not missing and not unexpected, f"checkpoint mismatch: missing={missing[:5]} unexpected={unexpected[:5]}"
    return b

# ---------------- sanity: a correctly loaded model is far better than chance ----------------
_probe = build_backbone(keep_head=True).to(DEVICE).eval()
_ids = tokenizer(clip(SP["train"]["pool"][0]["text"], 20000), add_special_tokens=False)["input_ids"]
if ARCH == 'lm':
    _t = torch.tensor([_ids[:512]], device=DEVICE)
    with torch.no_grad(), torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
        _loss = _probe(_t, targets=_t)[1].item()
    print(f"LM sanity loss on one case: {_loss:.3f}  (expect roughly 2-4; chance = {math.log(VOCAB_SIZE):.1f})")
    assert _loss < 6.0, "LM loss is too high - weights or tokenizer do not match"
else:
    _ph = PredHead(dict(HEAD_SD), _probe.embeddings.word_embeddings.weight.detach())
    if _ph.ok:
        ids_ = _ids[:509]
        P_ = min(320, int(0.63 * len(ids_))); S_ = len(ids_) - P_
        seq = [CLS_L] + ids_[:P_] + [SEP_L] + ids_[P_:] + [SEP_L]
        T_, Lp = len(seq), P_ + 2
        idx = torch.arange(T_, device=DEVICE)
        pm = ((idx[None, :] < Lp) | (idx[None, :] <= idx[:, None])).long()[None]     # prefix bidirectional, suffix causal
        tt = torch.tensor([[0] * Lp + [1] * (T_ - Lp)], device=DEVICE)
        try:
            with torch.no_grad(), torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
                hs = _probe(input_ids=torch.tensor([seq], device=DEVICE), attention_mask=pm, token_type_ids=tt).last_hidden_state
                lg = _ph(hs[:, Lp - 1:T_ - 1])
            tgt = torch.tensor([seq[Lp:]], device=DEVICE)
            _loss = F.cross_entropy(lg.float().reshape(-1, VOCAB_SIZE), tgt.reshape(-1)).item()
            print(f"continuation sanity loss ({S_} suffix tokens): {_loss:.3f}  (chance = {math.log(VOCAB_SIZE):.2f}; a trained model is clearly lower)")
            assert _loss < 8.0, "continuation loss is near chance - weights or tokenizer do not match"
        except AssertionError:
            raise
        except Exception as e:
            print("sanity check skipped (3-D attention mask not supported by this transformers version):", repr(e)[:200])
    else:
        print("prediction-head weights not in the checkpoint - sanity check skipped (only the encoder is needed)")
    del _ph
del _probe, _ids; gc.collect()
if DEVICE.type == 'cuda': torch.cuda.empty_cache()
print(f"{MODEL_NAME} loaded OK: {sum(v.numel() for v in ENC_SD.values()) / 1e6:.1f}M tensor elements in the checkpoint")

# ---------------- token encoders + document encoder ----------------
class LMTokenEncoder(nn.Module):
    # Models 4/6: EfficientLegalLM used as a bidirectional encoder: (ids, mask) -> final-norm hidden states
    def __init__(self, backbone, bidirectional, bidir_window, use_gc):
        super().__init__()
        self.backbone, self.bidirectional, self.bidir_window, self.use_gc = backbone, bidirectional, bidir_window, use_gc
    def forward(self, ids, mask):
        m = self.backbone; T = ids.size(1); pm = mask.bool()
        x = m.tok_emb(ids)
        cos, sin = m.freqs_cos[:T], m.freqs_sin[:T]
        window = self.bidir_window if self.bidirectional else WINDOW_SIZE
        full, win = build_masks(T, pm, self.bidirectional, window)
        for blk in m.layers:
            mk = win if blk.attn.use_window else full
            if self.use_gc and self.training and torch.is_grad_enabled():
                x = grad_checkpoint(block_forward, blk, x, cos, sin, mk, use_reentrant=False)
            else:
                x = block_forward(blk, x, cos, sin, mk)
        return m.final_norm(x)

class BertTokenEncoder(nn.Module):
    # Model 7: bidirectional BertModel
    def __init__(self, enc):
        super().__init__(); self.enc = enc
    def forward(self, ids, mask):
        return self.enc(input_ids=ids, attention_mask=mask).last_hidden_state

class DocEncoder(nn.Module):
    # case -> windows -> token encoder -> per-window vector -> aggregate over windows -> L2-normalised case embedding
    def __init__(self, token_encoder, hidden, pad_id, pooling):
        super().__init__()
        self.encoder, self.pad_id, self.pooling = token_encoder, pad_id, pooling
        if pooling == 'attention':
            self.att = nn.Sequential(nn.Linear(hidden, hidden // 2), nn.Tanh(), nn.Linear(hidden // 2, 1))
    def forward(self, docs):
        flat = [w for d in docs for w in d]; n = len(flat); L = max(len(w) for w in flat)
        ids = np.full((n, L), self.pad_id, dtype=np.int64); mask = np.zeros((n, L), dtype=np.int64)
        for i, w in enumerate(flat):
            ids[i, :len(w)] = w; mask[i, :len(w)] = 1
        ids = torch.from_numpy(ids).to(DEVICE); mask = torch.from_numpy(mask).to(DEVICE)
        hs = self.encoder(ids, mask)
        if WINDOW_POOL == 'mean':
            m = mask.unsqueeze(-1).to(hs.dtype); e = (hs * m).sum(1) / m.sum(1).clamp(min=1)
        else:
            e = hs[:, 0]
        e = e.float(); out = []; p = 0
        for d in docs:
            k = len(d); x = e[p:p + k]; p += k
            if self.pooling == 'attention':
                a = torch.softmax(self.att(x).float().squeeze(-1), 0); v = (a.unsqueeze(-1) * x).sum(0)
            else:
                v = x.mean(0)
            out.append(v)
        return F.normalize(torch.stack(out), dim=-1)

def pick_layers(layers):
    chosen = np.linspace(0, len(layers) - 1, STUDENT_LAYERS).round().astype(int)
    return nn.ModuleList([layers[int(i)] for i in chosen])

def make_model(method='full', student=False, verbose=True):
    # method in {full, lora, lorma}; student=True keeps STUDENT_LAYERS evenly spaced pre-trained layers (knowledge distillation)
    assert method in ('full', 'lora', 'lorma')
    base = build_backbone()
    if student:
        if ARCH == 'lm':
            base.layers = pick_layers(base.layers); base.n_layers = STUDENT_LAYERS
        else:
            base.encoder.layer = pick_layers(base.encoder.layer)
            for new_i, layer in enumerate(base.encoder.layer):          # re-index attention modules after shortening the stack
                for mod in layer.modules():
                    if hasattr(mod, 'layer_idx'): mod.layer_idx = new_i
            base.config.num_hidden_layers = STUDENT_LAYERS
    if method in ('lora', 'lorma'):
        for p in base.parameters():
            p.requires_grad_(False)
        n_ad = adapt_linears(base, method, RANK, ALPHA) if ARCH == 'lm' else adapt_bert(base, method, RANK, ALPHA)
        if verbose: print(f"{method}: wrapped {n_ad} linear layers")
    if ARCH == 'lm':
        te = LMTokenEncoder(base, BIDIRECTIONAL, BIDIR_WINDOW, USE_GC)
    else:
        if USE_GC:
            base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        te = BertTokenEncoder(base)
    model = DocEncoder(te, D_MODEL, PAD_L, POOLING).to(DEVICE)
    if verbose:
        tr = sum(p.numel() for p in model.parameters() if p.requires_grad); tot = sum(p.numel() for p in model.parameters())
        print(f"{MODEL_NAME}/{method}{' (student)' if student else ''} trainable / total params: {tr:,} / {tot:,} ({tr / tot:.1%})")
    return model

def trainable_state(model):
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in names}

def load_ckpt_into(model, payload):
    missing, unexpected = model.load_state_dict(payload['state'], strict=False)
    assert not unexpected, unexpected[:10]
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    assert not (set(missing) & names), list(set(missing) & names)[:10]
    return model

## 7. Encoding, dev evaluation, training loop (InfoNCE, optional distillation)
For `knowledge_distillation` the teacher's embeddings of all training queries / candidates are computed once (teacher on GPU, then freed) and cached on CPU; the student then trains on `KD_HARD_WEIGHT * InfoNCE + (1-KD_HARD_WEIGHT) * (KL(student || teacher in-batch score distribution) + cosine distance to the teacher embeddings)`.

In [ ]:
@torch.no_grad()
def encode_all(model, kind, records, bs=INFER_DOCS):
    model.eval(); out = []
    for i in tqdm(range(0, len(records), bs), desc=f"encode {kind}", leave=False):
        docs = [WZ.get(kind, r["id"]) for r in records[i:i + bs]]
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP):
            e = model(docs)
        out.append(e.float().cpu())
    return torch.cat(out).numpy()

def dense_scores(model, split):
    sp = SP[split]; kq, kc = {"dev": ("dq", "dc"), "test": ("tq", "tc")}[split]
    Eq = encode_all(model, kq, sp["queries"]); Ec = encode_all(model, kc, sp["pool"])
    return Eq @ Ec.T

def sample_negs(qid, rng):
    pool = HN[qid]
    if len(pool) >= HN_PER_Q:
        return [pool[i] for i in rng.choice(len(pool), HN_PER_Q, replace=False)]
    cids = SP["train"]["cids"]; out = list(pool)
    while len(out) < HN_PER_Q:
        c = cids[rng.randint(len(cids))]
        if c not in GOLD_SET[qid] and c != qid and (("c", c) in WZ.cache):
            out.append(c)
    return out

def cleanup():
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

def fit(method, name, student=False, teacher=None):
    # method: full | lora | lorma (the KD student is trained with method='full', student=True, teacher=<cached teacher embeddings>)
    model = make_model(method, student)
    enc_p = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith('encoder.')]
    head_p = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith('encoder.')]
    lr = LR_BACKBONE[method]
    groups = ([{'params': head_p, 'lr': HEAD_LR}] if head_p else []) + [{'params': enc_p, 'lr': lr}]
    opt = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)
    n_tr = len(TR_Q); spe = math.ceil(n_tr / BATCH_Q); total = math.ceil(spe / GRAD_ACCUM) * EPOCHS
    sched = get_linear_schedule_with_warmup(opt, int(total * WARMUP_FRAC), total)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    rng = np.random.RandomState(SEED)
    best = -1.0; bad = 0; path = OUTPUT_DIR / f'best_{name}_pcr.pt'; t_start = time.time(); stop = False
    for epoch in range(EPOCHS):
        model.train(); opt.zero_grad(set_to_none=True); order = rng.permutation(n_tr); losses = []
        for bi in tqdm(range(spe), desc=f"{name} epoch {epoch+1}/{EPOCHS}", leave=False):
            qs = [TR_Q[i] for i in order[bi * BATCH_Q:(bi + 1) * BATCH_Q]]
            B = len(qs)
            if B < 2:
                continue
            pos = [str(rng.choice(GOLD_IN_POOL[r["id"]])) for r in qs]
            negs = [c for r in qs for c in sample_negs(r["id"], rng)]
            dids = pos + negs
            q_w = [WZ.get("q", r["id"]) for r in qs]; d_w = [WZ.get("c", c) for c in dids]
            madd = torch.zeros(B, len(dids))
            for i, r in enumerate(qs):
                g = GOLD_SET[r["id"]]
                for j, c in enumerate(dids):
                    if j != i and c in g:
                        madd[i, j] = -1e4          # another relevant doc of this query must not act as a negative
            madd = madd.to(DEVICE)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP):
                emb = model(q_w + d_w)
            q, d = emb[:B].float(), emb[B:].float()
            logits = (q @ d.T) / TEMP + madd
            loss = F.cross_entropy(logits, torch.arange(B, device=DEVICE))
            if teacher is not None:                # knowledge distillation from the cached teacher embeddings
                tq = torch.from_numpy(np.stack([teacher['q'][r["id"]] for r in qs])).float().to(DEVICE)
                td = torch.from_numpy(np.stack([teacher['c'][c] for c in dids])).float().to(DEVICE)
                t_logits = (tq @ td.T) / TEMP + madd
                kl = F.kl_div(F.log_softmax(logits, dim=1), F.softmax(t_logits, dim=1), reduction='batchmean')
                cos = (1 - (q * tq).sum(-1)).mean() + (1 - (d * td).sum(-1)).mean()
                loss = KD_HARD_WEIGHT * loss + (1 - KD_HARD_WEIGHT) * (kl + cos)
            scaler.scale(loss / GRAD_ACCUM).backward(); losses.append(loss.item())
            if (bi + 1) % GRAD_ACCUM == 0 or bi == spe - 1:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            if TIME_BUDGET_MIN and (time.time() - t_start) / 60 > TIME_BUDGET_MIN:
                stop = True; print(f"time budget of {TIME_BUDGET_MIN} min reached -> stopping training"); break
        te = time.time()
        S = dense_scores(model, "dev"); f1, P, R = score_f1(S, SP["dev"])
        print(f"{name} epoch {epoch+1} loss {np.mean(losses):.4f} | dev dense F1@7 {f1:.4f} P {P:.4f} R {R:.4f} | eval {time.time()-te:.0f}s | elapsed {(time.time()-t_start)/60:.0f}m")
        if f1 > best + 1e-6:
            best = f1; bad = 0
            torch.save({'state': trainable_state(model), 'model_option': MODEL_OPTION, 'method': method, 'student': student,
                        'pooling': POOLING, 'window_pool': WINDOW_POOL, 'representation': REPRESENTATION,
                        'max_len': MAX_LEN, 'chunks': CHUNKS, 'best_dev_f1': best}, path)
        else:
            bad += 1
            if bad >= PATIENCE:
                print("Early stopping"); break
        if stop:
            break
    del model, opt, sched, scaler; cleanup()
    return path, best

def load_best(path):
    payload = torch.load(path, map_location='cpu', weights_only=False)
    assert payload['model_option'] == MODEL_OPTION, f"checkpoint is for Model {payload['model_option']}, notebook is set to Model {MODEL_OPTION}"
    return load_ckpt_into(make_model(payload['method'], payload['student'], verbose=False), payload)

def teacher_embeddings(path):
    # teacher on GPU -> embeddings of every training query / candidate used in training -> cached on CPU -> teacher freed
    model = load_best(path)
    recs_c = [r for r in SP["train"]["pool"] if r["id"] in NEED_C]
    Eq = encode_all(model, "q", TR_Q); Ec = encode_all(model, "c", recs_c)
    del model; cleanup()
    return {'q': {r["id"]: Eq[i].astype(np.float16) for i, r in enumerate(TR_Q)},
            'c': {r["id"]: Ec[i].astype(np.float16) for i, r in enumerate(recs_c)}}

## 8. Candidate registry: hybrid weight tuning, prediction writer
Every candidate (each single model/method and the all-model ensemble) is scored on dev with the best dense + TF-IDF weight; whichever has the best dev Micro-F1@7 writes `prediction.json` / `submission.zip` immediately.

In [ ]:
W_GRID = [0.0, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0] if USE_HYBRID else [0.0]
BEST = {'name': None, 'f1': -1.0}
RESULTS = {}
DENSE = {}   # name -> {'dev': S, 'test': S}  (raw cosine matrices)

def write_prediction(S_test):
    sp = SP["test"]; top = topk_ids(S_test, sp); pool = set(sp["cids"]); pred = {}
    for i, row in enumerate(top):
        ids = [sp["cids"][j] for j in row]
        assert len(ids) == TOPK == len(set(ids)) and all(x in pool for x in ids) and sp["queries"][i]["id"] not in ids
        pred[sp["queries"][i]["id"]] = ids
    assert len(pred) == len(sp["queries"])
    out = OUTPUT_DIR / "prediction.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(pred, f, ensure_ascii=False, indent=2)
    with zipfile.ZipFile(OUTPUT_DIR / "submission.zip", "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(out, arcname="prediction.json")

def search_hybrid(Zd):
    best = (-1.0, 0.0, 0.0, 0.0)
    for w in W_GRID:
        f1, P, R = score_f1(Zd + w * LEXZ["dev"], SP["dev"])
        if f1 > best[0] + 1e-9:
            best = (f1, P, R, w)
    return best

def consider_z(name, Zd, Zt, extra=None):
    # Zd / Zt: row-z-scored dense score matrices (dev / test) of this candidate
    f1, P, R, w = search_hybrid(Zd)
    d0 = score_f1(Zd, SP["dev"])[0]
    RESULTS[name] = {'dev_f1_dense': float(d0), 'dev_f1_final': float(f1), 'dev_P7': float(P), 'dev_R7': float(R), 'hybrid_w': float(w), **(extra or {})}
    print(f"[{name}] dev F1@7 dense-only {d0:.4f} | best hybrid w={w} -> F1@7 {f1:.4f} (P {P:.4f}, R {R:.4f})" + (f" | {extra}" if extra else ""))
    if f1 > BEST['f1']:
        BEST.update(name=name, f1=float(f1))
        write_prediction(Zt + w * LEXZ["test"])
        print(f"   -> new best ({name}); prediction.json + submission.zip written")
    with open(OUTPUT_DIR / "results.json", "w") as fh:
        json.dump({'results': RESULTS, 'best': BEST,
                   'config': {'MODEL_OPTION': MODEL_OPTION, 'RUN_METHODS': RUN_METHODS, 'POOLING': POOLING,
                              'REPRESENTATION': REPRESENTATION, 'CHUNKS': CHUNKS, 'MAX_LEN': MAX_LEN}}, fh, indent=2)

def consider(name, S_dev, S_test, extra=None):
    consider_z(name, zrow(S_dev), zrow(S_test), extra)

# lexical-only candidate so there is always a valid submission
write_prediction(LEX["test"]); BEST.update(name='tfidf', f1=float(score_f1(LEX["dev"], SP["dev"])[0])); RESULTS['tfidf'] = {'dev_f1_final': BEST['f1']}
print("fallback tfidf prediction written | dev F1@7", round(BEST['f1'], 4))

## 9. Train / evaluate
Runs every method in `RUN_METHODS` for the selected model in the order full -> lora -> lorma -> knowledge_distillation (one model in memory at a time). `knowledge_distillation` reuses the `full` checkpoint of this run as its teacher; if `full` was not run, a full teacher is trained first (`best_<model>_kd_teacher_pcr.pt`).

In [ ]:
TEACHER_PATH = None

def existing_ckpt(name):
    c = Path(CKPT_OVERRIDE.get(name) or (OUTPUT_DIR / f'best_{name}_pcr.pt'))
    return c if ((not RETRAIN) and c.exists()) else None

def train_or_reuse(method, name):
    p = existing_ckpt(name)
    if p is not None:
        print(f"[{name}] RETRAIN=False -> using {p}"); return p
    return fit(method, name)[0]

def run_one(method):
    global TEACHER_PATH
    t_m = time.time()
    name = f"{PFX}_{'kd' if method == 'knowledge_distillation' else method}"
    if method == 'knowledge_distillation':
        path = existing_ckpt(name)
        if path is not None:
            print(f"[{name}] RETRAIN=False -> using {path}")
        else:
            if TEACHER_PATH is None:
                TEACHER_PATH = train_or_reuse('full', f'{PFX}_kd_teacher')
            teacher = teacher_embeddings(TEACHER_PATH)
            path = fit('full', name, student=True, teacher=teacher)[0]
            del teacher; cleanup()
    else:
        path = train_or_reuse(method, name)
        if method == 'full':
            TEACHER_PATH = path
    model = load_best(path)
    t_enc = time.time(); S_dev = dense_scores(model, "dev"); enc_time = time.time() - t_enc
    S_test = dense_scores(model, "test")
    np.save(OUTPUT_DIR / f"dev_scores_{name}.npy", S_dev.astype(np.float16)); np.save(OUTPUT_DIR / f"test_scores_{name}.npy", S_test.astype(np.float16))
    DENSE[name] = {'dev': S_dev, 'test': S_test}
    consider(name, S_dev, S_test)
    RESULTS[name]['dev_encode_seconds'] = round(enc_time, 1); RESULTS[name]['total_minutes'] = round((time.time() - t_m) / 60, 1)
    del model; cleanup()

for method in [m for m in METHOD_ORDER if m in RUN_METHODS]:
    print(f"\n===== {MODEL_NAME} / {method.upper()} =====")
    run_one(method)

## 10. Ensemble of all models trained in this session, final summary

In [ ]:
if len(DENSE) >= 2:
    Sd = np.mean([zrow(v['dev']) for v in DENSE.values()], 0); St = np.mean([zrow(v['test']) for v in DENSE.values()], 0)
    consider_z('ensemble(' + '+'.join(DENSE) + ')', Sd, St)

print("\nSummary (dev):")
for k, v in sorted(RESULTS.items(), key=lambda kv: -kv[1].get('dev_f1_final', 0)):
    print(f"  {k:46s} F1@7 {v.get('dev_f1_final', 0):.4f}" + (f" | dense-only {v['dev_f1_dense']:.4f} | w_lex={v['hybrid_w']}" if 'hybrid_w' in v else ""))
print("\nSubmitted:", BEST['name'], "| dev F1@7", round(BEST['f1'], 4))
with open(OUTPUT_DIR / "prediction.json") as f:
    pred = json.load(f)
k0 = next(iter(pred)); print("queries:", len(pred), "| sample:", k0, pred[k0])
assert len(pred) == len(SP["test"]["queries"]) and all(len(v) == TOPK == len(set(v)) for v in pred.values())
try:
    from IPython.display import display, FileLink
    display(FileLink(str(OUTPUT_DIR / "prediction.json"))); display(FileLink(str(OUTPUT_DIR / "submission.zip")))
except Exception:
    pass

## 11. Notes
- **Runtime:** each model trains up to `EPOCHS` (early-stopped by `PATIENCE`). Running all four methods trains up to five models sequentially (full, lora, lorma, KD teacher is the `full` run, KD student). Set `TIME_BUDGET_MIN`, lower `EPOCHS`, or run methods in separate sessions (`RUN_METHODS=['full']`, then `['lora']`, ... - each session writes its own `dev_scores_*.npy` / `test_scores_*.npy`, which can be re-ensembled offline). To reuse a trained model in a later session set `RETRAIN=False` and point `CKPT_OVERRIDE` at the saved `best_<name>_pcr.pt` (names are `m<N>_full`, `m<N>_lora`, `m<N>_lorma`, `m<N>_kd`, `m<N>_kd_teacher`).
- **Choosing the model:** change `MODEL_OPTION` (4, 6 or 7) and attach that model's `best.pt` and tokenizer; each model writes to its own `pcr_p7_model<N>` folder. Model 6 is the heaviest; Model 7 uses a 512-token position limit (`MAX_LEN=384` fits).
- **OOM (T4):** lower `BATCH_Q` first (then `CHUNKS` / `MAX_LEN`). Gradient checkpointing is on (`USE_GC`).
- **Knowledge distillation:** the student keeps `STUDENT_LAYERS` evenly spaced layers of the pre-trained model (alternating window/full pattern of Models 4/6 is preserved per layer). The teacher is never on the GPU while the student trains; its embeddings for the training queries and candidates are cached on CPU. Validation / model selection always uses the real dev labels.
- **LoRA / LoRMA** use the Phase-3 definitions (LoRMA = multiplicative output-space update, `B` starts at zero, so training starts from the pre-trained model). For Model 7 the adapters are placed on q/k/v, attention output, intermediate and output dense layers of every BERT layer (as in the Phase-6 notebook).
- The hybrid TF-IDF weight grid was extended to 8.0 because the previous run picked the old upper edge (3.0).